# Day 4 — Alpha & Beta (OLS) vs Nifty 100

Compute **Alpha** and **Beta** for each fund using **scipy.stats.linregress**:

- Regression (daily): `fund_daily_return ~ alpha + beta * nifty_100_daily_return`
- **Beta** = slope
- **Alpha (annualized, as requested)** = `intercept * 252`

Outputs:
- `Data/processed/alpha_beta.csv`

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import linregress


In [2]:
# --- Repo-root detection (so notebook works from any working directory) ---
HERE = Path(__file__).resolve() if '__file__' in globals() else Path.cwd()

def _find_repo_root(start: Path) -> Path:
    cand = start
    for _ in range(12):
        if (cand / 'Data' / 'processed' / 'daily_returns_all_schemes.csv').exists() and (cand / 'Data' / 'processed' / 'benchmark_indices_clean.csv').exists():
            return cand
        if cand.name == 'notebooks':
            parent = cand.parent
            if (parent / 'Data' / 'processed' / 'daily_returns_all_schemes.csv').exists() and (parent / 'Data' / 'processed' / 'benchmark_indices_clean.csv').exists():
                return parent
        cand = cand.parent
    return start.parent

REPO_ROOT = _find_repo_root(HERE)
DATA_DIR = REPO_ROOT / 'Data' / 'processed'

print('DATA_DIR:', DATA_DIR.resolve())


DATA_DIR: C:\Mutual Fund Analytics\Data\processed


In [3]:
# --- Input paths ---
daily_returns_path = DATA_DIR / 'daily_returns_all_schemes.csv'
bench_path = DATA_DIR / 'benchmark_indices_clean.csv'
fund_path = DATA_DIR / 'fund_master_clean.csv'

for p in [daily_returns_path, bench_path, fund_path]:
    if not p.exists():
        raise FileNotFoundError(f'Missing: {p}')

TRADING_DAYS = 252
min_obs = 30  # minimum daily overlap for regression


In [4]:
# --- Load data ---
returns_df = pd.read_csv(daily_returns_path)
bench_df = pd.read_csv(bench_path)
fund_df = pd.read_csv(fund_path)

returns_df['date'] = pd.to_datetime(returns_df['date'], errors='coerce')
returns_df['amfi_code'] = pd.to_numeric(returns_df['amfi_code'], errors='coerce').astype('Int64')
returns_df['daily_return'] = pd.to_numeric(returns_df['daily_return'], errors='coerce')

if 'scheme_name' not in returns_df.columns:
    fund_df['amfi_code'] = pd.to_numeric(fund_df['amfi_code'], errors='coerce').astype('Int64')
    fund_df = fund_df.dropna(subset=['amfi_code', 'scheme_name']).copy()
    returns_df = returns_df.merge(fund_df[['amfi_code', 'scheme_name']], on='amfi_code', how='left')

fund_df['amfi_code'] = pd.to_numeric(fund_df['amfi_code'], errors='coerce').astype('Int64')
name_map = dict(zip(fund_df['amfi_code'].dropna().astype(int), fund_df['scheme_name']))

bench_df['date'] = pd.to_datetime(bench_df['date'], errors='coerce')
bench_df = bench_df.dropna(subset=['date']).copy()

returns_df = returns_df.dropna(subset=['date', 'amfi_code', 'daily_return']).copy()

print('schemes:', int(returns_df['amfi_code'].nunique()))
print('returns rows:', len(returns_df))
print('bench columns:', bench_df.columns.tolist())


schemes: 40
returns rows: 64280
bench columns: ['date', 'index_name', 'close_value']


In [5]:
# --- Build Nifty 100 daily returns from benchmark_indices_clean.csv ---
# Expected schema in this repo: date, index_name, close_value
required_bench_cols = {'date', 'index_name', 'close_value'}
missing = required_bench_cols - set(bench_df.columns)
if missing:
    raise ValueError(f'benchmark_indices_clean.csv missing columns: {sorted(missing)}')

idx_name = bench_df['index_name'].astype(str).str.lower()
mask = idx_name.str.contains('nifty') & idx_name.str.contains('100')
bench_nifty100 = bench_df.loc[mask, ['date', 'close_value']].copy()

if bench_nifty100.empty:
    raise ValueError('Could not find Nifty 100 in benchmark_indices_clean.csv (index_name contains both nifty and 100).')

bench_nifty100 = bench_nifty100.sort_values('date')
bench_nifty100['close_value'] = pd.to_numeric(bench_nifty100['close_value'], errors='coerce')
bench_nifty100 = bench_nifty100.dropna(subset=['close_value']).copy()

bench_nifty100['nifty_100_return'] = bench_nifty100['close_value'].pct_change()
bench_series = bench_nifty100.dropna(subset=['nifty_100_return']).copy()

print('Nifty 100 daily rows:', len(bench_series))


Nifty 100 daily rows: 1149


In [6]:
# --- Alpha/Beta computation per fund ---
returns_df = returns_df.sort_values(['amfi_code', 'date']).copy()

returns_small = returns_df[['date', 'amfi_code', 'scheme_name', 'daily_return']].copy()

bench_map = bench_series.set_index('date')['nifty_100_return']

rows = []
fund_codes = sorted(returns_small['amfi_code'].dropna().astype(int).unique().tolist())

for code in fund_codes:
    fr = returns_small[returns_small['amfi_code'].astype(int) == int(code)].dropna(subset=['daily_return']).copy()
    if fr.empty:
        continue

    merged = fr.merge(bench_map, left_on='date', right_index=True, how='inner')
    if len(merged) < min_obs:
        continue

    x = merged['nifty_100_return'].astype(float).values
    y = merged['daily_return'].astype(float).values

    lr = linregress(x, y)

    beta = float(lr.slope)
    alpha_annual = float(lr.intercept) * float(TRADING_DAYS)  # requested

    scheme_name = merged['scheme_name'].iloc[0] if 'scheme_name' in merged.columns and pd.notna(merged['scheme_name'].iloc[0]) else name_map.get(int(code), str(code))

    rows.append({
        'amfi_code': int(code),
        'scheme_name': scheme_name,
        'alpha_annual': alpha_annual,
        'beta': beta,
        'n_obs': int(len(merged)),
    })

alpha_beta_df = pd.DataFrame(rows)
print('computed rows:', len(alpha_beta_df))

out_path = DATA_DIR / 'alpha_beta.csv'
alpha_beta_df.to_csv(out_path, index=False)
print('wrote:', out_path.resolve())


computed rows: 40
wrote: C:\Mutual Fund Analytics\Data\processed\alpha_beta.csv


In [7]:
alpha_beta_df.sort_values('alpha_annual', ascending=False).head(10)


,amfi_code,scheme_name,alpha_annual,beta,n_obs
21,119598,SBI Small Cap Fund - Regular Plan - Growth,0.303370,-0.023196,1149
39,149324,DSP Small Cap Fund - Regular - Growth,0.300579,0.011455,1149
25,120505,ICICI Pru Midcap Fund - Regular - Growth,0.292636,0.000549,1149
36,148569,Mirae Asset Tax Saver Fund - Regular - Growth,0.282704,0.018134,1149
30,120843,Kotak Flexicap Fund - Regular - Growth,0.273305,-0.022830,1149
2,100033,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,0.271954,0.005104,1149
34,148567,Mirae Asset Large Cap Fund - Regular - Growth,0.269838,0.023684,1149
38,149323,DSP Midcap Fund - Regular - Growth,0.265986,-0.002523,1149
16,119094,Axis Midcap Fund - Regular - Growth,0.260767,-0.066265,1149
19,119551,SBI Bluechip Fund - Regular Plan - Growth,0.232010,-0.031751,1149
